# 05. Workstation training runner

This notebook runs the whole training pipeline by calling
`scripts/train_experiments.py`. It contains **no training logic of its own**:
every model, grid, fold and metric lives in the script, so this notebook and a
plain command line produce the same results. If something here disagrees with
the script, the script is right.

## How to run

One command, from the project root:

```
jupyter nbconvert --to notebook --execute notebooks/05_workstation_training.ipynb --ExecutePreprocessor.timeout=-1 --output 05_workstation_training.ipynb
```

`--ExecutePreprocessor.timeout=-1` removes the per-cell time limit, which
matters because section 3 can run for well over an hour. The executed notebook,
with all output, is written back over itself, so it is the run's record.

Or open it in Jupyter and use **Run All**. Either way, edit section 0 first if
the defaults do not suit the machine.

## If it stops

Nothing is lost. Every unit writes its own checkpoint the moment it finishes,
so **re-running the same command resumes** and skips what is already done.

| symptom | what to do |
| --- | --- |
| a cell raised, section 3 reports failed units | read the per-unit tail it prints, fix the cause, run again; the units that succeeded are skipped |
| the run stopped itself with a memory message | close other applications, or lower `PARALLEL_UNITS`, and run again |
| section 4 says a CV value is not finite | that unit failed silently in an older version; rerun it with `--force --only Model/feature_set` |
| section 5 says notebook 04 failed | the training is still fine, the checkpoints are on disk; rerun this notebook, which re-executes only section 5's work |
| you want to start over | delete the checkpoint folder printed in section 1 and run again |

Nothing in this notebook deletes or overwrites a finished checkpoint unless you
set `FORCE = True` in section 0.

## 0. Parameters

The only cell meant to be edited. The defaults suit a 16-core workstation with
64 GB of RAM.

In [ ]:
# Which grid to search. "workstation" widens the XGBoost and Random Forest
# grids and puts max_depth=None back; "standard" is the laptop grid. Each
# profile writes to its own checkpoint folder, so the two can be compared.
PROFILE = "workstation"

# How many units run at the same time, each in its own process.
PARALLEL_UNITS = 3

# Threads inside each unit. PARALLEL_UNITS x N_JOBS_PER_UNIT should not exceed
# the core count; 3 x 5 = 15 on 16 cores leaves one core for the operating
# system and for this notebook.
N_JOBS_PER_UNIT = 5

# The learning curve, the out-of-fold residual quantiles and the permutation
# importance. Each is its own resumable unit and none touches the nine main
# units.
RUN_EXTRAS = True

# Smoke mode: 3-fold CV, a tenth of the rows, single-value grids, 50 trees,
# written to reports/checkpoints_smoke. It proves the pipeline executes and
# nothing else. Leave this False for a real run.
SMOKE = False

# Redo units that already have a checkpoint. Leave False to resume.
FORCE = False

# Memory floors: a unit waits rather than starting a fit below these, a minute
# at a time for up to 30 minutes, then stops cleanly. None uses the profile
# default, which is 0.4 GB RAM and 1 GB commit headroom for the standard
# profile and 4 GB and 8 GB for the workstation profile, because an unbounded
# forest in the wider grid is over 1 GB on its own.
#
# Lower them only on a machine where the fits are known to be small. A smoke
# run on a laptop is the case that needs it: two Python interpreters alone can
# take the commit headroom below 1 GB, and the guard would then correctly
# refuse to start fits that would in fact have been harmless.
MIN_RAM_GB = 0.15 if SMOKE else None
MIN_COMMIT_GB = 0.3 if SMOKE else None

# Also run notebook 04 to build the tables and figures.
RUN_RESULTS_NOTEBOOK = True

## 1. Preflight

Everything that would make the run fail, checked before anything expensive
starts. A missing input stops the notebook here with the path it expected,
rather than thirty minutes in.

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SCRIPT = PROJECT_ROOT / "scripts" / "train_experiments.py"
RESULTS_NOTEBOOK = PROJECT_ROOT / "notebooks" / "04_model_training.ipynb"

# A smoke run writes model_results_smoke.json and figures_smoke/, so it cannot
# overwrite a real result. Notebook 04 applies the same suffix.
suffix = "_smoke" if SMOKE else ""

RUN_STARTED = time.perf_counter()
problems: list[str] = []
warnings: list[str] = []

print("=" * 78)
print("PREFLIGHT")
print("=" * 78)

# --- environment ---------------------------------------------------------
print()
print("environment")
print(f"  python            : {platform.python_version()} ({sys.executable.split(os.sep)[-1]})")
print(f"  platform          : {platform.system()} {platform.release()}")
# As a named tuple value rather than a literal, so the check survives a
# linter that knows this project targets 3.11 and calls the branch dead.
MINIMUM_PYTHON = (3, 11)
if sys.version_info < MINIMUM_PYTHON:
    problems.append(
        f"Python {platform.python_version()} is older than "
        f"{'.'.join(str(v) for v in MINIMUM_PYTHON)}, which this project targets"
    )

PACKAGES = ("numpy", "pandas", "scikit-learn", "xgboost", "joblib", "psutil",
            "matplotlib", "seaborn", "pyarrow", "nbconvert", "nbformat")
missing_packages = []
for name in PACKAGES:
    try:
        from importlib.metadata import version

        print(f"  {name:<18}: {version(name)}")
    except Exception:
        missing_packages.append(name)
        print(f"  {name:<18}: MISSING")
if missing_packages:
    problems.append(
        "missing packages: " + ", ".join(missing_packages)
        + " (install with: pip install -r requirements-dev.txt)"
    )

# --- machine -------------------------------------------------------------
import psutil  # noqa: E402

cores_logical = os.cpu_count() or 1
cores_physical = psutil.cpu_count(logical=False) or cores_logical
memory = psutil.virtual_memory()
disk = shutil.disk_usage(PROJECT_ROOT)

print()
print("machine")
print(f"  cores             : {cores_physical} physical, {cores_logical} logical")
print(f"  RAM total         : {memory.total / 1e9:.1f} GB")
print(f"  RAM available     : {memory.available / 1e9:.1f} GB")
print(f"  disk free         : {disk.free / 1e9:.1f} GB")

thread_budget = PARALLEL_UNITS * N_JOBS_PER_UNIT
print()
print("plan")
print(f"  profile           : {PROFILE}{'  (SMOKE)' if SMOKE else ''}")
print(f"  parallel units    : {PARALLEL_UNITS}")
print(f"  threads per unit  : {N_JOBS_PER_UNIT}")
print(f"  peak threads      : {thread_budget}")
print(f"  extras            : {RUN_EXTRAS}")
print(f"  force             : {FORCE}")
print(f"  RAM floor         : "
      f"{MIN_RAM_GB if MIN_RAM_GB is not None else 'profile default'}")
print(f"  commit floor      : "
      f"{MIN_COMMIT_GB if MIN_COMMIT_GB is not None else 'profile default'}")

if thread_budget > cores_logical:
    warnings.append(
        f"PARALLEL_UNITS x N_JOBS_PER_UNIT = {thread_budget} exceeds "
        f"{cores_logical} logical cores, so units will contend for CPU and the "
        f"run will be slower than the plan suggests"
    )
if not SMOKE and PROFILE == "workstation" and memory.total < 30e9:
    warnings.append(
        f"the workstation profile puts max_depth=None back in the Random Forest "
        f"grid, where one forest can exceed 1 GB; this machine has "
        f"{memory.total / 1e9:.1f} GB of RAM, so consider PROFILE = 'standard' "
        f"or PARALLEL_UNITS = 1"
    )
if disk.free < 5e9:
    warnings.append(f"only {disk.free / 1e9:.1f} GB of disk free; the fitted "
                    f"pipelines for one profile can reach a few hundred MB")

# --- inputs --------------------------------------------------------------
print()
print("inputs")
REQUIRED = {
    "training script": SCRIPT,
    "dataset": PROJECT_ROOT / "data" / "processed" / "master_listings.parquet",
    "column sets": PROJECT_ROOT / "reports" / "column_sets.json",
    "run 1 probe record": PROJECT_ROOT / "reports" / "model_results_v1_sqrt.json",
    "results notebook": RESULTS_NOTEBOOK,
}
WHERE_FROM = {
    "dataset": "rebuild it by running notebooks/03_build_master_dataset.ipynb",
    "column sets": "written by notebooks/03_build_master_dataset.ipynb",
    "run 1 probe record": "the stored max_features probe; the script reads the "
                          "rule-selected 0.33 from it rather than rerunning it",
}
for name, path in REQUIRED.items():
    relative = path.relative_to(PROJECT_ROOT)
    if path.exists():
        print(f"  OK      {name:<20} {relative}  ({path.stat().st_size / 1e6:.2f} MB)")
    else:
        print(f"  MISSING {name:<20} {relative}")
        hint = WHERE_FROM.get(name, "")
        problems.append(
            f"{name} not found at {relative}" + (f"; {hint}" if hint else "")
        )

# --- the dataset itself --------------------------------------------------
if REQUIRED["dataset"].exists():
    import pandas as pd

    master = pd.read_parquet(REQUIRED["dataset"], engine="pyarrow")
    in_scope = int((master["long_stay"] == False).sum())  # noqa: E712
    print()
    print("dataset")
    print(f"  rows              : {len(master):,}")
    print(f"  columns           : {len(master.columns)}")
    print(f"  long_stay == False: {in_scope:,}  (the modelling population)")
    if in_scope < 100:
        problems.append(
            f"only {in_scope} rows with long_stay == False; the dataset looks wrong"
        )
    del master

# --- where the output will go -------------------------------------------
import scripts.train_experiments as te  # noqa: E402

config = te.configure(
    profile=PROFILE,
    smoke=SMOKE,
    min_ram_gb=MIN_RAM_GB,
    min_commit_gb=MIN_COMMIT_GB,
)
CHECKPOINT_DIR = config.checkpoints
EXPERIMENT_DIR = config.experiments

print()
print("output")
print(f"  configuration     : {config.describe()}")
print(f"  checkpoints       : reports/{CHECKPOINT_DIR.name}")
print(f"  pipelines         : models_store/{EXPERIMENT_DIR.name}")
if CHECKPOINT_DIR.exists():
    existing = sorted(p.name for p in CHECKPOINT_DIR.glob("*.json"))
    strays = sorted(p.name for p in CHECKPOINT_DIR.glob("*.tmp"))
    print(f"  existing files    : {len(existing)}")
    for name in existing:
        print(f"      {name}")
    if strays:
        warnings.append(
            f"stray temporary files in reports/{CHECKPOINT_DIR.name}: "
            f"{', '.join(strays)}; they are left by an interrupted unit and are "
            f"ignored, but you may delete them"
        )
else:
    print("  existing files    : none, this folder will be created")

# --- verdict -------------------------------------------------------------
print()
print("=" * 78)
for warning in warnings:
    print(f"WARNING: {warning}")
if problems:
    for problem in problems:
        print(f"PROBLEM: {problem}")
    print("=" * 78)
    raise SystemExit(
        f"preflight failed with {len(problems)} problem(s), listed above. "
        f"Nothing was run. Fix them and execute this notebook again."
    )
print(f"preflight passed with {len(warnings)} warning(s)")
print("=" * 78)

## 2. Plan

The script's own dry run, so the list of units below is the list it will
actually execute. Nothing is fitted by this cell.

In [ ]:
def script_command(*flags: str) -> list[str]:
    """The script, with the flags this notebook's parameters imply."""
    command = [sys.executable, str(SCRIPT), "--profile", PROFILE]
    if SMOKE:
        command.append("--smoke")
    if RUN_EXTRAS:
        command.append("--extras")
    if MIN_RAM_GB is not None:
        command += ["--min-ram-gb", str(MIN_RAM_GB)]
    if MIN_COMMIT_GB is not None:
        command += ["--min-commit-gb", str(MIN_COMMIT_GB)]
    return command + list(flags)


def run_script(*flags: str, stream: bool = True) -> tuple[int, list[str]]:
    """
    Run the script and show its output as it arrives.

    Streaming rather than collecting matters for section 3: the run takes
    long enough that a silent cell is indistinguishable from a hung one.
    """
    command = script_command(*flags)
    print("$ python " + " ".join(
        str(Path(part).name) if part.endswith(".py") else part
        for part in command[1:]
    ))
    print("-" * 78)
    lines: list[str] = []
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        cwd=str(PROJECT_ROOT),
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    for line in process.stdout:
        line = line.rstrip()
        lines.append(line)
        if stream:
            print(line, flush=True)
    process.stdout.close()
    return process.wait(), lines


plan_code, plan_lines = run_script(
    "--dry-run",
    "--parallel-units", str(PARALLEL_UNITS),
    "--n-jobs", str(N_JOBS_PER_UNIT),
    *(["--force"] if FORCE else []),
)
print("-" * 78)
if plan_code != 0:
    raise SystemExit(
        f"the dry run exited {plan_code}, so the flags above are wrong. "
        f"Nothing was fitted. Check section 0."
    )
print("plan accepted")

## 3. Run

The training itself. Each unit runs in its own process, `PARALLEL_UNITS` at a
time, in two waves: the nine searches first, then the leakage check and the
extras, which read the searches' checkpoints to find out which model won.

Output streams in live. A unit that fails does not stop the others, and the
summary at the end names the ones to rerun.

**This cell is the long one.** It is also the resumable one: if it stops,
running it again skips every unit that finished.

In [ ]:
print("=" * 78)
print("TRAINING")
print("=" * 78)
train_started = time.perf_counter()

train_code, train_lines = run_script(
    "--parallel-units", str(PARALLEL_UNITS),
    "--n-jobs", str(N_JOBS_PER_UNIT),
    *(["--force"] if FORCE else []),
)

train_minutes = (time.perf_counter() - train_started) / 60
print("-" * 78)
print(f"elapsed: {train_minutes:.1f} minutes")

if train_code != 0:
    # Say what survived, what did not, and where to look, then stop. The
    # checkpoints are the ground truth here, not the exit code.
    print()
    print("=" * 78)
    print(f"TRAINING EXITED {train_code}")
    print("=" * 78)

    done, pending = [], []
    for model_name, set_name in te.UNITS:
        label = f"{model_name}/{set_name}"
        (done if te.checkpoint_path(model_name, set_name).exists() else pending).append(label)
    if te.checkpoint_path(*te.LEAKAGE_UNIT).exists():
        done.append("leakage/all")
    else:
        pending.append("leakage/all")
    for name in te.EXTRA_UNITS:
        label = f"extras/{name}"
        (done if te.extra_path(name).exists() else pending).append(label)

    print()
    print(f"finished ({len(done)}):")
    for label in done:
        print(f"  OK      {label}")
    print(f"not finished ({len(pending)}):")
    for label in pending:
        print(f"  PENDING {label}")

    log_path = PROJECT_ROOT / "reports" / "training_progress.log"
    if log_path.exists():
        tail = log_path.read_text(encoding="utf-8").splitlines()[-40:]
        print()
        print(f"last {len(tail)} lines of reports/{log_path.name}:")
        for line in tail:
            print(f"  {line}")

    print()
    print("Re-running this cell resumes. The units listed as finished have "
          "checkpoints and will be skipped, so only the pending ones are "
          "attempted again. Nothing above needs to be cleaned up first.")
    raise SystemExit(f"training exited {train_code}; see the summary above")

print()
print("training finished, every unit reported success")

## 4. Verify and backfill

Three steps: check every checkpoint, repair any CV figure that is not finite,
then check again. The assertion at the end is the gate. It is what stops a run
that failed quietly from being mistaken for a run that worked, which is exactly
what happened before `error_score="raise"` was added to the script.

A backfill recomputes a CV mean and SD at a unit's stored parameters. It does
not re-select parameters, so it repairs a missing number and not a bad search.

In [ ]:
print("=" * 78)
print("VERIFY")
print("=" * 78)
verify_code, _ = run_script("--verify-only")

print()
print("=" * 78)
print("BACKFILL (only units whose CV figures are not finite)")
print("=" * 78)
backfill_code, _ = run_script(
    "--backfill-cv", "--n-jobs", str(N_JOBS_PER_UNIT)
)
if backfill_code not in (0, 1):
    raise SystemExit(f"the backfill exited {backfill_code}, which is a failure")

print()
print("=" * 78)
print("VERIFY AGAIN")
print("=" * 78)
final_code, final_lines = run_script("--verify-only")

# The script is the single source of truth, but assert here too so the
# notebook itself fails loudly rather than printing a bad table and moving on.
reports = [
    te.inspect_checkpoint(model_name, set_name)
    for model_name, set_name in [*te.UNITS, te.LEAKAGE_UNIT]
]
missing = [r["unit"] for r in reports if not r["exists"]]
not_finite = [r["unit"] for r in reports if r["exists"] and not r["cv_finite"]]
incomplete = [r["unit"] for r in reports if not r["complete"]]

print()
print("assertions")
print(f"  units checked          : {len(reports)}")
print(f"  missing                : {missing or 'none'}")
print(f"  non-finite CV          : {not_finite or 'none'}")
print(f"  incomplete             : {incomplete or 'none'}")

assert not missing, (
    "these units have no checkpoint: " + ", ".join(missing)
    + ". Re-run section 3; it resumes."
)
assert not not_finite, (
    "these units have a non-finite CV mean or SD, which means a search failed "
    "silently: " + ", ".join(not_finite)
    + ". Re-run them with --force --only Model/feature_set."
)
assert not incomplete, (
    "these units are missing a predictions file or a fitted pipeline: "
    + ", ".join(incomplete) + ". Re-run them with --force."
)

extras_present = [n for n in te.EXTRA_UNITS if te.extra_path(n).exists()]
print(f"  extras present         : {extras_present or 'none'} "
      f"(optional, not asserted)")
print()
print("every required unit is complete and every CV figure is finite")

## 5. Results

Notebook 04 builds every table and figure from the checkpoints. It is executed
here as a subprocess rather than reimplemented, so there is one place where a
number is turned into a table and no chance of this notebook and that one
disagreeing.

It receives the profile through the environment, so a workstation run renders
the workstation checkpoints.

In [ ]:
results_table = None
results_ok = False

if not RUN_RESULTS_NOTEBOOK:
    print("RUN_RESULTS_NOTEBOOK is False, skipping notebook 04")
else:
    print("=" * 78)
    print("RESULTS NOTEBOOK (04_model_training.ipynb)")
    print("=" * 78)
    command = [
        sys.executable, "-m", "nbconvert",
        "--to", "notebook",
        "--execute",
        "--inplace",
        "--ExecutePreprocessor.timeout=-1",
        str(RESULTS_NOTEBOOK),
    ]
    print("$ python -m nbconvert --to notebook --execute --inplace "
          "--ExecutePreprocessor.timeout=-1 notebooks/04_model_training.ipynb")
    print("-" * 78)
    results_started = time.perf_counter()
    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        cwd=str(PROJECT_ROOT),
        env={**os.environ, **te.CONFIG.as_env(), "PYTHONUNBUFFERED": "1"},
    )
    print(completed.stdout[-4000:] if completed.stdout else "(no stdout)")
    if completed.returncode != 0:
        print("-" * 78)
        print("NOTEBOOK 04 FAILED")
        print("-" * 78)
        print(completed.stderr[-6000:])
        print()
        print("The training itself is fine: the checkpoints verified in "
              "section 4 are on disk and nothing above is lost. This is a "
              "rendering failure. Open notebooks/04_model_training.ipynb and "
              "run it cell by cell to see which cell raised, then re-run this "
              "notebook.")
        raise SystemExit(
            f"notebook 04 exited {completed.returncode}; the checkpoints are "
            f"intact, see the message above"
        )
    results_ok = True
    print("-" * 78)
    print(f"notebook 04 executed in "
          f"{(time.perf_counter() - results_started) / 60:.1f} minutes")

# The headline table, read from the file notebook 04 assembled.
import pandas as pd  # noqa: E402

model_results_path = PROJECT_ROOT / "reports" / f"model_results{suffix}.json"
if results_ok and model_results_path.exists():
    payload = json.loads(model_results_path.read_text(encoding="utf-8"))
    results_table = pd.DataFrame(payload["results"])
    columns = [c for c in (
        "model", "feature_set", "cv_rmse_log_mean", "cv_rmse_log_sd",
        "rmse_aud_typical", "mae_aud_typical", "r2_typical",
    ) if c in results_table.columns]
    print()
    print(f"=== results, profile {payload.get('profile', PROFILE)} ===")
    print(results_table[columns].round(4).to_string(index=False))
    print()
    print("winners, selected on CV log RMSE")
    for set_name, winner in payload.get("winners", {}).items():
        print(f"  {set_name:<16} {winner['model']:<14} "
              f"CV log RMSE {winner['cv_rmse_log_mean']:.4f}   "
              f"typical MAE {winner['mae_aud_typical']:,.2f}")
elif results_ok:
    print()
    print(f"notebook 04 ran but reports/{model_results_path.name} is not there, "
          f"which should not happen; open notebook 04 and check its section 10")

## 6. Finish

What to copy back, and how long it took.

In [ ]:
print("=" * 78)
print("DONE")
print("=" * 78)

OUTPUTS = [
    (f"reports/{CHECKPOINT_DIR.name}", CHECKPOINT_DIR,
     "one JSON per unit: parameters, CV figures, metrics, importances"),
    (f"reports/model_results{suffix}.json", model_results_path,
     "the assembled result, built by notebook 04"),
    (f"reports/figures{suffix}", PROJECT_ROOT / "reports" / f"figures{suffix}",
     "the figures notebook 04 saved"),
    (f"models_store/{EXPERIMENT_DIR.name}", EXPERIMENT_DIR,
     "the fitted pipelines, one per unit"),
    ("reports/training_progress.log", PROJECT_ROOT / "reports" / "training_progress.log",
     "the run log, with memory at every step"),
    ("notebooks/04_model_training.ipynb", RESULTS_NOTEBOOK,
     "the executed results notebook, with all output"),
]

print()
print("copy these back from the workstation:")
print()
for label, path, description in OUTPUTS:
    if not path.exists():
        print(f"  MISSING  {label}")
        print(f"           {description}")
        continue
    if path.is_dir():
        files = [p for p in path.rglob("*") if p.is_file()]
        size = sum(p.stat().st_size for p in files)
        print(f"  {label:<40} {len(files):>3} files  {size / 1e6:>8.1f} MB")
    else:
        print(f"  {label:<40} {'':>9}  {path.stat().st_size / 1e6:>8.1f} MB")
    print(f"      {description}")

print()
print(f"profile          : {PROFILE}{'  (SMOKE, results are not meaningful)' if SMOKE else ''}")
print(f"parallel units   : {PARALLEL_UNITS} x {N_JOBS_PER_UNIT} threads")
print(f"extras           : {RUN_EXTRAS}")
print(f"total runtime    : {(time.perf_counter() - RUN_STARTED) / 60:.1f} minutes")
print()
if SMOKE:
    print("This was a smoke run. Delete reports/checkpoints_smoke and "
          "models_store/experiments_smoke, set SMOKE = False, and run again "
          "for real results.")
else:
    print("Nothing here is staged or committed. reports/ and models_store/ are "
          "gitignored, so copy the folders above off the machine by hand.")